# 00 · Build the China A-share daily panel (the paper's market)

Two free Qlib-format dumps, both unpacked under `D:/ML_Markowitz_data/raw_cn`:

* **joshuaxql/qlib_data** (Hugging Face, `cn_data.zip`, SHA-256 checked): raw daily prices, VWAP, adjustment factor,
  total and float market cap, price limits, ST intervals, Shenwan level-1 industries (31, as in the paper),
  point-in-time financial indicators with announcement dates, 2010 onward, delisted stocks included.
* **chenditc/investment_data** (GitHub, `qlib_bin.tar.gz`): adjusted prices back to 2000, used for 2007-2009 history so
  long-horizon characteristics exist in 2010, and the CSI 500 index.

| | Paper | Here |
|---|---|---|
| Universe | A-shares, excluding ST and suspended stocks | Same; ST and suspension (no bar or zero volume) on decision day t |
| Target | VWAP(t+1) to VWAP(t+2) return minus the CSI 500 | Same; the CSI 500 leg uses its close-to-close return over day t+2 |
| Characteristics | 415 daily | 47 price/volume, 163 point-in-time financial indicators, 10 valuation ratios |
| Industries | 31 Shenwan dummies | 31 Shenwan level-1 dummies, membership as of day t |

A financial value is usable from the trading day **after** its announcement. Each characteristic is ranked across
stocks within each day to [-1, 1], with missing values set to 0, and stored as int8. Building the panel takes about
15-30 minutes and 6-8 GB of RAM; it is written to D:.

In [1]:
from pathlib import Path

import pandas as pd

from mlm import cn
from mlm.data import panel_features

RAW = Path("D:/ML_Markowitz_data/raw_cn")
J_ROOT, C_ROOT = RAW / "joshuaxql" / "cn_data", RAW / "chenditc" / "qlib_bin"
UNRANKED_DIR = Path("D:/ML_Markowitz_data/panel_cn_unranked")   # raw characteristics, kept for later checks
PANEL_DIR = Path("D:/ML_Markowitz_data/panel_cn")
KAGGLE_DIR = Path("D:/ML_Markowitz_data/kaggle_stockchina/original_data")   # optional independent check (JoinQuant data)
PANEL_START, PANEL_END = "2010-01-01", "2023-12-31"

Step 1: price/volume characteristics, target and universe for all stocks (skipped if already saved).

In [2]:
if not (UNRANKED_DIR / "meta.json").exists():
    panel, numeric, dummies = cn.build_panel(J_ROOT, C_ROOT, PANEL_DIR, PANEL_START, PANEL_END)
    cn.save_unranked(panel, numeric, dummies, UNRANKED_DIR)
    del panel

5490 stocks


4155 trading days, 2007-01-04 to 2024-01-31


source check on the first 20 days of 2010: 32,813 stock-days in both, return corr 1.0000, median |diff| 5.7e-05


tradability: 212,119 stock-days dropped as listed < 60 trading days, 11,033 as locked at the upper limit on t+1


universe: 10,043,676 stock-days, 2,953 stocks per day


...

  high52: 94.7% non-missing


  uphits21: 98.9% non-missing


  dnhits21: 98.9% non-missing


  uphits252: 92.5% non-missing


industry known for 98.1% of rows


Step 2: add point-in-time fundamentals and valuation ratios, rank within each day, one year at a time. Years already written are skipped.

In [3]:
field_names = cn.field_names_from_manifest(RAW / "joshuaxql" / "manifest.json")
features = cn.write_cn_panel(UNRANKED_DIR, J_ROOT, PANEL_DIR, field_names)
print(len(features), "features in total")

fundamentals: 226,896 announcement events, 163 indicators


2010: 372,105 rows, 242 days, 1,538 stocks/day, fundamentals coverage 67%


2011: 452,813 rows, 244 days, 1,856 stocks/day, fundamentals coverage 97%


2012: 522,378 rows, 243 days, 2,150 stocks/day, fundamentals coverage 98%


2013: 547,883 rows, 238 days, 2,302 stocks/day, fundamentals coverage 98%


...

2019: 844,669 rows, 244 days, 3,462 stocks/day, fundamentals coverage 99%


2020: 879,012 rows, 243 days, 3,617 stocks/day, fundamentals coverage 99%


2021: 982,722 rows, 243 days, 4,044 stocks/day, fundamentals coverage 99%


2022: 1,086,024 rows, 242 days, 4,488 stocks/day, fundamentals coverage 99%


2023: 1,163,088 rows, 242 days, 4,806 stocks/day, fundamentals coverage 99%
251 features in total


## Sanity checks: stocks per day by year (paper's Table A.1) and target distribution

In [4]:
u = pd.concat([pd.read_parquet(f, columns=["date", "target"]) for f in sorted(UNRANKED_DIR.glob("*.parquet"))])
paper_a1 = {2010: 1841, 2011: 2169, 2012: 2393, 2013: 2462, 2014: 2527, 2015: 2736, 2016: 2932, 2017: 3345,
            2018: 3612, 2019: 3741, 2020: 4015, 2021: 4485, 2022: 4869, 2023: 5196}
counts = u.groupby("date").size()
pd.DataFrame({"Avg stocks per day": counts.groupby(counts.index.year).mean().round(0),
              "Paper Table A.1": pd.Series(paper_a1), "Days": counts.groupby(counts.index.year).size()})

,Avg stocks per day,Paper Table A.1,Days
2010,1538.0,1841,242
2011,1856.0,2169,244
2012,2150.0,2393,243
2013,2302.0,2462,238
2014,2246.0,2527,245
2015,2205.0,2736,244
2016,2519.0,2932,244
2017,2871.0,3345,244
2018,3249.0,3612,243
2019,3462.0,3741,244


In [5]:
print(f"{len(u):,} stock-days in total (paper text: 5,962,768; its Table A.1 implies about 11.2 million)")
u["target"].describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999])

10,043,676 stock-days in total (paper text: 5,962,768; its Table A.1 implies about 11.2 million)


count    1.004368e+07
mean     1.171817e-04
std      2.456381e-02
min     -2.918741e-01
0.1%    -1.102660e-01
1%      -6.404237e-02
50%     -1.115680e-03
99%      7.798643e-02
99.9%    1.317329e-01
max      3.729460e-01
Name: target, dtype: float64

## Independent check of the VWAP returns against the Kaggle StockChina files

In [6]:
cn.crosscheck_kaggle(J_ROOT, KAGGLE_DIR) if KAGGLE_DIR.exists() else "Kaggle folder not found, skipped"

stocks compared           277.000000
stock-days             479108.000000
return corr                 0.999529
median |diff|               0.000308
share |diff| < 0.1%         0.858994
share |diff| < 1%           0.999937
dtype: float64